In [0]:
dbutils.widgets.text("batch_id", "92199a9d-c132-4525-92c9-5094e8e1f310")
dbutils.widgets.text("run_date", "2026-10-08")
dbutils.widgets.dropdown("load_type", "FULL", ["FULL", "INCREMENTAL"])
dbutils.widgets.text("manifest_path", "")


In [0]:
import json
import uuid
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

spark.conf.set("spark.sql.session.timeZone", "UTC")
batch_id = dbutils.widgets.get("batch_id").strip()
run_date = dbutils.widgets.get("run_date").strip()
load_type = dbutils.widgets.get("load_type").strip().upper()
manifest_override = dbutils.widgets.get("manifest_path").strip()

assert load_type == "FULL", "This QA test expects FULL historical input"
assert datetime.strptime(run_date, "%Y-%m-%d").strftime("%Y-%m-%d") == run_date
assert batch_id, "batch_id cannot be empty"
root = Path("/Volumes/github_health/raw/bronze_data/bronze")
manifest_path = (Path(manifest_override) if manifest_override else
                 root / "_manifests" / f"full_load_{run_date}_{batch_id}.json")
assert manifest_path.is_file(), f"Missing manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest.get("batch_id") == batch_id
assert manifest.get("run_date", run_date) == run_date
assert manifest.get("load_type", "FULL") == "FULL"
expected = {(r, e) for r in ["apache/spark", "pandas-dev/pandas", "scikit-learn/scikit-learn"]
            for e in ["commits", "issues", "pulls"]}
entries = [x for x in manifest["files"] if (x.get("repository"), x.get("entity")) in expected]
assert len(entries) == 9 and {(x["repository"], x["entity"]) for x in entries} == expected
for item in entries:
    assert Path(item["path"]).is_file(), f"Missing source JSONL: {item['path']}"
print("PASS 1: Parameterized FULL manifest and 9 JSONL files verified", flush=True)
print("Manifest:", manifest_path)

# Capture production checkpoints, but never write to their table.
wm_table = "github_health.ops.pipeline_watermarks"
def read_production_watermarks():
    return [tuple(r) for r in (
        spark.table(wm_table)
        .filter(F.col("layer") == "RAW")
        .select("repository", "entity", "layer", "last_successful_watermark", "batch_id", "updated_at")
        .orderBy("repository", "entity", "layer")
        .collect()
    )]
watermarks_before = read_production_watermarks()

TABLE = "github_health.ops.qa_historical_backfill"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {TABLE} (
    record_key STRING,
    repository STRING,
    issue_state STRING,
    source_updated_at TIMESTAMP,
    payload_hash STRING,
    load_timestamp TIMESTAMP,
    batch_id STRING
) USING DELTA""")
SCHEMA = StructType([
    StructField("record_key", StringType(), False),
    StructField("repository", StringType(), False),
    StructField("issue_state", StringType(), True),
    StructField("source_updated_at", TimestampType(), True),
    StructField("payload_hash", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
    StructField("batch_id", StringType(), True),
])
key = "QA_BACKFILL|" + uuid.uuid4().hex
full_end = datetime.fromisoformat(manifest["window_end"].replace("Z", "+00:00"))
full_end = full_end.astimezone(timezone.utc).replace(tzinfo=None)
old_event = full_end - timedelta(days=1)
new_event = full_end + timedelta(hours=1)
old_processing = full_end + timedelta(hours=2)
new_processing = full_end + timedelta(hours=3)
replay_processing = full_end + timedelta(hours=4)

def merge_qa(state, event_time, process_time, source_batch):
    content = json.dumps({"state": state, "updated_at": event_time.isoformat()}, sort_keys=True)
    digest = hashlib.sha256(content.encode("utf-8")).hexdigest()
    df = spark.createDataFrame([(
        key, "apache/spark", state, event_time, digest, process_time, source_batch
    )], schema=SCHEMA)
    values = {c: f"s.`{c}`" for c in df.columns}
    updates = {c: v for c, v in values.items() if c != "record_key"}
    # Critical protection: a different hash alone is insufficient to allow an older replay.
    (DeltaTable.forName(spark, TABLE).alias("t")
     .merge(df.alias("s"), "t.record_key = s.record_key")
     .whenMatchedUpdate(
         condition=(
             "NOT (t.payload_hash <=> s.payload_hash) "
             "AND s.source_updated_at IS NOT NULL "
             "AND (t.source_updated_at IS NULL OR s.source_updated_at > t.source_updated_at)"
         ),
         set=updates
     )
     .whenNotMatchedInsert(values=values)
     .execute())

def stored():
    rows = spark.table(TABLE).filter(F.col("record_key") == key).collect()
    assert len(rows) == 1, f"Expected one QA record, got {len(rows)}"
    return rows[0]

# Historical version inserted.
merge_qa("OPEN", old_event, old_processing, batch_id)
first = stored()
assert first["issue_state"] == "OPEN" and first["load_timestamp"] == old_processing
print("PASS 2: Historical record inserted")

# GitHub publishes a genuinely newer change.
merge_qa("CLOSED", new_event, new_processing, "qa_newer_incremental")
latest = stored()
assert latest["issue_state"] == "CLOSED"
assert latest["source_updated_at"] == new_event
assert latest["load_timestamp"] == new_processing
assert latest["payload_hash"] != first["payload_hash"]
print("PASS 3: Newer issue update applied")

# The older FULL batch is replayed, but must not overwrite the newer record.
merge_qa("OPEN", old_event, replay_processing, batch_id)
after_stale = stored()
assert after_stale["issue_state"] == "CLOSED"
assert after_stale["source_updated_at"] == new_event
assert after_stale["load_timestamp"] == new_processing
assert after_stale["batch_id"] == "qa_newer_incremental"
print("PASS 4: Stale historical replay could not overwrite newer version")

# A new identical replay must also leave timestamps untouched.
merge_qa("CLOSED", new_event, replay_processing, "qa_newer_incremental")
after_same = stored()
assert after_same["load_timestamp"] == new_processing
assert after_same["payload_hash"] == latest["payload_hash"]
print("PASS 5: Identical replay left load_timestamp unchanged")

count = spark.table(TABLE).filter(F.col("record_key") == key).count()
assert count == 1
assert read_production_watermarks() == watermarks_before
print("PASS 6: One QA row; production RAW checkpoints unchanged")
print("\nALL 6 QA HISTORICAL BACKFILL TESTS PASSED")
print("Full-load batch:", batch_id)
print("Final QA state:", after_same["issue_state"])
print("Final QA row count:", count)
print("Production Bronze/Silver tables were not changed")
